<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# Dynamic Column Mapping — Input & Output Showcase

Demonstrates how to use `column_mapping` (input) and `configure_output` (output) to work with **any DataFrame schema** without renaming columns manually.

**Extractor used:** `CoverageExtractor` — the simplest extractor, requiring only `id` and `geometry`.

**Key concepts:**
- **Input mapping** (`column_mapping`) — tells the extractor which DataFrame columns correspond to internal fields (`id`, `geometry`, `crop`, `start_date`, …)
- **Output rename** (`output_mapping`) — rename result columns before export
- **Output exclude** (`exclude_columns`) — drop unwanted columns from results
- **Output select** (`output_columns`) — whitelist only the columns you need

## Step 0: Bootstrap

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

## Step 1: Initialisation

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager

manager = WorkflowManager("prod", log_to_console=False, log_level="WARNING")

## Step 2: Get entities

We build **two DataFrames** with different schemas to demonstrate that column mapping adapts to any naming convention.

- **Scenario A** — Platform data (columns like `crop.id`, `sowingDate`)
- **Scenario B** — External / custom data (columns like `parcel_code`, `wkt`, `culture`)

### Scenario A — Platform-style DataFrame

Columns use the EarthDaily Agriculture platform naming (`crop.id`, `sowingDate`, etc.).

In [ ]:
import pandas as pd

# Simulates a DataFrame loaded from the EarthDaily Agriculture platform
# Note: column names do NOT match the internal canonical names (id, geometry, crop, start_date, end_date)
platform_entities = pd.DataFrame([
    {
        "id": "field_001",
        "geometry": "POLYGON((-58.9454 -13.7203, -58.9422 -13.7317, -58.9281 -13.7303, -58.9316 -13.7189, -58.9454 -13.7203))",
        "crop.id": "CORN",
        "sowingDate": "2025-01-15",
        "endDate": "2025-06-15",
        "name": "North Field",
        "field.farm.grower.firstname": "John",
    },
    {
        "id": "field_002",
        "geometry": "POLYGON((-47.0600 -22.9000, -47.0580 -22.9020, -47.0560 -22.9000, -47.0580 -22.8980, -47.0600 -22.9000))",
        "crop.id": "SOYBEAN",
        "sowingDate": "2025-02-01",
        "endDate": "2025-07-01",
        "name": "South Field",
        "field.farm.grower.firstname": "Maria",
    },
])

print("Scenario A — Platform entity columns:")
print(list(platform_entities.columns))
display(platform_entities)

### Scenario B — External / custom DataFrame

Columns use completely custom names from an external dataset (shapefile, client CSV, etc.).

In [ ]:
# Simulates a DataFrame from an external source — no column matches the internal names at all
external_entities = pd.DataFrame([
    {
        "parcel_code": "EXT-A1",
        "wkt": "POLYGON((-58.9454 -13.7203, -58.9422 -13.7317, -58.9281 -13.7303, -58.9316 -13.7189, -58.9454 -13.7203))",
        "culture": "CORN",
        "date_debut": "2025-01-15",
        "date_fin": "2025-06-15",
        "proprietaire": "Carlos",
        "surface_ha": 120.5,
    },
    {
        "parcel_code": "EXT-B2",
        "wkt": "POLYGON((-47.0600 -22.9000, -47.0580 -22.9020, -47.0560 -22.9000, -47.0580 -22.8980, -47.0600 -22.9000))",
        "culture": "SOYBEAN",
        "date_debut": "2025-02-01",
        "date_fin": "2025-07-01",
        "proprietaire": "Ana",
        "surface_ha": 85.3,
    },
])

print("Scenario B — External entity columns:")
print(list(external_entities.columns))
display(external_entities)

## Step 3: Input Column Mapping

The `column_mapping` dict tells the extractor how to find internal fields in **your** DataFrame.

```
column_mapping = {
    "internal_name": "your_column_name",
    ...
}
```

Only map fields that differ from the defaults. Fields already matching (`id`, `geometry`, etc.) don't need mapping.

### Available internal field names

| Internal key | Default column | Purpose |
|---|---|---|
| `id` | `id` | Entity identifier |
| `geometry` | `geometry` | WKT polygon |
| `crop` | `crop` | Crop type code |
| `start_date` | `start_date` | Extraction period start |
| `end_date` | `end_date` | Extraction period end |
| `sowing_date` | `sowing_date` | Crop sowing date |

### 3A — Mapping for platform data

Platform returns `crop.id` and `sowingDate` — only those two need mapping. `id` and `geometry` already match the defaults.

In [ ]:
from earthdaily.agriculture.extractors.coverage_function import CoverageExtractor

# --- Scenario A: Platform data ---
# Only map fields whose column names differ from the internal defaults
platform_mapping = {
    "crop": "crop.id",          # internal "crop" -> DataFrame column "crop.id"
    "start_date": "sowingDate",  # internal "start_date" -> DataFrame column "sowingDate"
    "end_date": "endDate",       # internal "end_date" -> DataFrame column "endDate"
}

extractor_a = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor_a.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    column_mapping=platform_mapping,  # <-- input mapping applied here
)

# Verify: the extractor now knows how to resolve fields
print("Active column mapping:")
for internal, mapped in extractor_a.column_mapping.items():
    marker = " <-- custom" if internal in platform_mapping else ""
    print(f"  {internal:15s} -> {mapped}{marker}")

### 3B — Mapping for external / custom data

Every column name is different — we need to map `id`, `geometry`, `crop`, `start_date`, and `end_date`.

In [ ]:
# --- Scenario B: External data — every field needs mapping ---
external_mapping = {
    "id": "parcel_code",        # internal "id" -> DataFrame column "parcel_code"
    "geometry": "wkt",           # internal "geometry" -> DataFrame column "wkt"
    "crop": "culture",           # internal "crop" -> DataFrame column "culture"
    "start_date": "date_debut",  # internal "start_date" -> DataFrame column "date_debut"
    "end_date": "date_fin",      # internal "end_date" -> DataFrame column "date_fin"
}

extractor_b = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor_b.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    column_mapping=external_mapping,  # <-- input mapping applied here
)

print("Active column mapping:")
for internal, mapped in extractor_b.column_mapping.items():
    marker = " <-- custom" if internal in external_mapping else ""
    print(f"  {internal:15s} -> {mapped}{marker}")

### Validate mapping against your DataFrame

Use `validate_column_mapping()` to catch mismatches early — before the extraction starts.

In [ ]:
# Validate that the mapping matches the actual DataFrame columns
print("--- Scenario A: Platform data ---")
valid_a = extractor_a.validate_column_mapping(platform_entities)
print(f"Mapping valid: {valid_a}\n")

print("--- Scenario B: External data ---")
valid_b = extractor_b.validate_column_mapping(external_entities)
print(f"Mapping valid: {valid_b}\n")

# Demonstrate a bad mapping — intentional mismatch for illustration
print("--- Bad mapping example ---")
extractor_bad = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)
extractor_bad.setup_coverage_parameters(
    start_date="2025-01-01",
    column_mapping={"id": "wrong_column", "geometry": "also_wrong"},
)
valid_bad = extractor_bad.validate_column_mapping(external_entities)
print(f"Mapping valid: {valid_bad}  (expected False — warnings above show what's missing)")

### How the mapping resolves at runtime

`get_entity_value(row, key)` looks up the mapped column first, then falls back to the canonical key.

In [ ]:
# Show how the extractor resolves field values at runtime for both scenarios
row_a = platform_entities.iloc[0]
row_b = external_entities.iloc[0]

print("--- Scenario A: Platform row ---")
print(f"  get_entity_value(row, 'id')         -> {extractor_a.get_entity_value(row_a, 'id')}")
print(f"  get_entity_value(row, 'crop')       -> {extractor_a.get_entity_value(row_a, 'crop')}")
print(f"  get_entity_value(row, 'start_date') -> {extractor_a.get_entity_value(row_a, 'start_date')}")
print(f"  get_entity_value(row, 'end_date')   -> {extractor_a.get_entity_value(row_a, 'end_date')}")

print("\n--- Scenario B: External row ---")
print(f"  get_entity_value(row, 'id')         -> {extractor_b.get_entity_value(row_b, 'id')}")
print(f"  get_entity_value(row, 'crop')       -> {extractor_b.get_entity_value(row_b, 'crop')}")
print(f"  get_entity_value(row, 'start_date') -> {extractor_b.get_entity_value(row_b, 'start_date')}")
print(f"  get_entity_value(row, 'end_date')   -> {extractor_b.get_entity_value(row_b, 'end_date')}")

print("\nBoth extractors resolve the same internal fields from different column names.")

## Step 4: Extract — Single Entity

Run `process_single_entity_coverage` on both scenarios to confirm the mapping works end-to-end.

In [ ]:
# --- Scenario A: single entity from platform data ---
print("--- Scenario A: Platform entity ---")
result_a = extractor_a.process_single_entity_coverage(platform_entities.iloc[0])

if result_a["data"] is not None and not result_a["data"].empty:
    print(f"Rows returned: {len(result_a['data'])}")
    print(f"Output columns: {list(result_a['data'].columns)}")
    display(result_a["data"].head())
else:
    print(f"No data: {result_a.get('error')}")

In [ ]:
# --- Scenario B: single entity from external data ---
print("--- Scenario B: External entity ---")
result_b = extractor_b.process_single_entity_coverage(external_entities.iloc[0])

if result_b["data"] is not None and not result_b["data"].empty:
    print(f"Rows returned: {len(result_b['data'])}")
    print(f"Output columns: {list(result_b['data'].columns)}")
    display(result_b["data"].head())
else:
    print(f"No data: {result_b.get('error')}")

## Step 5: Output Formatting

Output formatting is applied during `_finalize_extraction()` (bulk) or can be called manually via `apply_output_format()`.

Three options, applied in order: **rename** -> **exclude** -> **select**.

| Parameter | Purpose | Example |
|---|---|---|
| `output_mapping` | Rename columns | `{"crop.id": "crop", "sowingDate": "sowing"}` |
| `exclude_columns` | Drop columns | `["field.farm.grower.firstname", "geometry"]` |
| `output_columns` | Whitelist — keep only these | `["id", "date", "coverage_percent", "sensor"]` |

### 5A — Output mapping: rename columns

Use `output_mapping` to rename result columns to your preferred names. The input metadata columns (from your entity DataFrame) flow through to the output — rename them here.

In [ ]:
# Create an extractor with output_mapping: rename platform columns in the result
extractor_rename = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor_rename.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    # Input mapping
    column_mapping={
        "crop": "crop.id",
        "start_date": "sowingDate",
        "end_date": "endDate",
    },
    # Output mapping — rename metadata columns that came from the entity DataFrame
    output_mapping={
        "crop.id": "crop",
        "sowingDate": "sowing_date",
        "endDate": "end_date",
        "field.farm.grower.firstname": "grower_name",
    },
)

# Run single entity extraction
result_rename = extractor_rename.process_single_entity_coverage(platform_entities.iloc[0])

if result_rename["data"] is not None and not result_rename["data"].empty:
    # apply_output_format() is called automatically in bulk; for single entity, call it manually
    df_renamed = extractor_rename.apply_output_format(result_rename["data"])
    print("Columns BEFORE rename:", list(result_rename["data"].columns))
    print("Columns AFTER rename: ", list(df_renamed.columns))
    display(df_renamed.head())
else:
    print(f"No data: {result_rename.get('error')}")

### 5B — Exclude columns

Use `exclude_columns` to drop unwanted columns (e.g. grower PII, verbose metadata).

In [ ]:
# Create an extractor with exclude_columns
extractor_exclude = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor_exclude.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    column_mapping={"crop": "crop.id", "start_date": "sowingDate", "end_date": "endDate"},
    # Exclude: drop grower PII and verbose name column from output
    exclude_columns=["field.farm.grower.firstname", "name", "sowingDate", "endDate"],
)

result_exclude = extractor_exclude.process_single_entity_coverage(platform_entities.iloc[0])

if result_exclude["data"] is not None and not result_exclude["data"].empty:
    df_raw = result_exclude["data"]
    df_excluded = extractor_exclude.apply_output_format(df_raw)
    print(f"Columns BEFORE exclude ({len(df_raw.columns)}):", list(df_raw.columns))
    print(f"Columns AFTER exclude  ({len(df_excluded.columns)}):", list(df_excluded.columns))
    display(df_excluded.head())
else:
    print(f"No data: {result_exclude.get('error')}")

### 5C — Output columns (whitelist)

Use `output_columns` to keep **only** the columns you need. Applied after rename, so use the renamed names.

In [ ]:
# Create an extractor combining rename + whitelist
extractor_select = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor_select.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    column_mapping={"crop": "crop.id", "start_date": "sowingDate", "end_date": "endDate"},
    # Rename first, then select — output_columns uses the RENAMED names
    output_mapping={"crop.id": "crop"},
    output_columns=["id", "crop", "date", "image_id", "coverage_percent", "sensor", "spatial_resolution"],
)

result_select = extractor_select.process_single_entity_coverage(platform_entities.iloc[0])

if result_select["data"] is not None and not result_select["data"].empty:
    df_raw = result_select["data"]
    df_selected = extractor_select.apply_output_format(df_raw)
    print(f"Columns BEFORE ({len(df_raw.columns)}): {list(df_raw.columns)}")
    print(f"Columns AFTER  ({len(df_selected.columns)}): {list(df_selected.columns)}")
    display(df_selected.head())
else:
    print(f"No data: {result_select.get('error')}")

## Step 6: Bulk Extraction with Full Mapping Pipeline

End-to-end example: external data with custom column names, input mapping, and output formatting — all in one bulk extraction. Output formatting is applied automatically during `_finalize_extraction()`.

In [ ]:
# Full pipeline: external data -> input mapping -> extraction -> output formatting
extractor_full = CoverageExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

extractor_full.setup_coverage_parameters(
    vegetation_index="NDVI",
    start_date="2025-01-01",
    clear_cover_min=80,
    filter="duplicate",
    mask="auto",
    # INPUT: map external column names to internal fields
    column_mapping={
        "id": "parcel_code",
        "geometry": "wkt",
        "crop": "culture",
        "start_date": "date_debut",
        "end_date": "date_fin",
    },
    # OUTPUT: rename columns for the final deliverable
    output_mapping={
        "parcel_code": "entity_id",
        "culture": "crop_type",
        "date_debut": "season_start",
        "date_fin": "season_end",
        "proprietaire": "owner",
    },
    # OUTPUT: drop unwanted columns
    exclude_columns=["surface_ha"],
)

In [ ]:
# Run bulk extraction on external entities
# Output formatting (rename + exclude) is applied automatically during _finalize_extraction()
results_full = extractor_full.process_entity_coverage_bulk_parallel(
    entity_list=external_entities,
    max_workers=5,
    output_path=manager.output_result_dir,
    skip_export=False,
    prefix="column_mapping_demo",
)

In [ ]:
# Inspect the final output
df_final = results_full["results_df"]

print(f"Final output shape: {df_final.shape}")
print(f"Final columns: {list(df_final.columns)}")
print(f"\nNote: 'parcel_code' was renamed to 'entity_id', 'culture' to 'crop_type', etc.")
print(f"      'surface_ha' was excluded from the output.")
display(df_final.head(10))

## Summary — Data Flow

```
Your DataFrame                      Extractor internals                   Final output
(any column names)                  (canonical names)                     (your preferred names)

parcel_code  ──column_mapping──>    id           ──API call──>           entity_id      (output_mapping)
wkt          ──column_mapping──>    geometry     ──API call──>           (excluded)
culture      ──column_mapping──>    crop         ──normalize──>          crop_type      (output_mapping)
date_debut   ──column_mapping──>    start_date   ──normalize──>          season_start   (output_mapping)
date_fin     ──column_mapping──>    end_date     ──normalize──>          season_end     (output_mapping)
proprietaire ────────────────────>  (passthrough) ──normalize──>          owner          (output_mapping)
surface_ha   ────────────────────>  (passthrough) ──normalize──>          (excluded)     (exclude_columns)
                                                   + API result columns:  date, image_id, coverage_percent, ...
```

**Key takeaways:**
- `column_mapping` is **input-side**: tells the extractor where to find `id`, `geometry`, etc. in your DataFrame
- `output_mapping`, `exclude_columns`, `output_columns` are **output-side**: control what the final DataFrame looks like
- Entity metadata columns (from your DataFrame) flow through to the output via `normalize_with_metadata()`
- Output formatting is applied automatically in bulk (`_finalize_extraction`); for single entity, call `apply_output_format()` manually
- All three output options can be combined: rename -> exclude -> select (applied in this order)